# 01 - Ingestion, data contract, and the Bronze landing zone
Rubric deliverable 1 (Ingestion) and the Bronze part of deliverable 2.

Run order: this notebook assumes `docker compose -f docker/docker-compose.yml up -d`
is running (Kafka on `localhost:9092`).

In [1]:
import os, sys
os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) == 'notebooks' else None
os.environ.setdefault('TQDM_DISABLE', '1')
print('cwd:', os.getcwd())

cwd: C:\Users\Huawei\clinical-vitals-capstone


## 1. The Pydantic data contract rejects malformed records
`validate_record` returns `(reading, [])` when valid, or `(None, [reasons])` when not.

In [2]:
from src.contracts.vitals import validate_record
from datetime import datetime, timezone, timedelta

good = dict(reading_id='11111111-1111-1111-1111-111111111111', patient_id='P100001',
            recorded_at=datetime.now(timezone.utc).isoformat(), heart_rate=78, resp_rate=16,
            systolic_bp=120, spo2=98, temperature_c=36.8, consciousness='A',
            on_supplemental_o2=False, source_device='GE-CARESCAPE-B450')
print('valid   ->', validate_record(good))

for mutate in [lambda r: r.update(heart_rate=400),
               lambda r: r.update(spo2=3),
               lambda r: r.update(consciousness='Z'),
               lambda r: r.update(diagnosis='sepsis'),
               lambda r: r.update(recorded_at=(datetime.now(timezone.utc)+timedelta(days=2)).isoformat())]:
    bad = dict(good); mutate(bad)
    reading, reasons = validate_record(bad)
    print('rejected ->', reasons)

valid   -> (VitalSignReading(reading_id='11111111-1111-1111-1111-111111111111', patient_id='P100001', recorded_at=datetime.datetime(2026, 9, 9, 16, 8, 19, 361604, tzinfo=TzInfo(UTC)), heart_rate=78, resp_rate=16, systolic_bp=120, spo2=98, temperature_c=36.8, consciousness=<Consciousness.ALERT: 'A'>, on_supplemental_o2=False, source_device='GE-CARESCAPE-B450'), [])
rejected -> ['heart_rate: Input should be less than or equal to 250']
rejected -> ['spo2: Input should be greater than or equal to 50']
rejected -> ["consciousness: Input should be 'A', 'V', 'P' or 'U'"]
rejected -> ['diagnosis: Extra inputs are not permitted']
rejected -> ['recorded_at: Value error, recorded_at 2026-09-11T16:08:19.361604+00:00 is in the future']


## 2. Generate a mixed batch (valid + deliberately malformed)

In [3]:
from src.generator.synth_vitals import generate, _summarise
records = generate(rows=600, bad_rate=0.15, seed=42)
print(_summarise(records))

total=600  valid=517  malformed=83
  malformed[bad_enum] = 4
  malformed[bad_patient_id] = 7
  malformed[empty_reading_id] = 11
  malformed[future_timestamp] = 10
  malformed[hr_out_of_range] = 10
  malformed[missing_required] = 4
  malformed[spo2_out_of_range] = 6
  malformed[structurally_broken] = 9
  malformed[temp_out_of_range] = 4
  malformed[unknown_field] = 11
  malformed[wrong_type] = 7


## 3. Produce to Kafka, consume with contract enforcement, route the results
Valid -> Delta **Bronze**. Malformed -> Kafka topic **`vitals.deadletter`** with the reason
and the source offset. Nothing malformed reaches Bronze.

In [4]:
import json, pathlib
from src.ingestion.admin import ensure_topics
from src.ingestion.producer import publish_file
from src.ingestion.consumer import run as consume
from src.lakehouse.bronze import read_bronze

pathlib.Path('data/raw').mkdir(parents=True, exist_ok=True)
with open('data/raw/nb_ingest.jsonl', 'w', encoding='utf-8') as fh:
    for r in records: fh.write(json.dumps(r) + '\n')

import shutil; shutil.rmtree('lakehouse/bronze', ignore_errors=True)
from kafka.admin import KafkaAdminClient
a = KafkaAdminClient(bootstrap_servers='localhost:9092')
try: a.delete_topics(['vitals.raw', 'vitals.deadletter'])
except Exception as e: print(e)
a.close()
import time; time.sleep(3)
ensure_topics()
publish_file(pathlib.Path('data/raw/nb_ingest.jsonl'))
stats = consume(max_messages=600, idle_timeout=15)
print(stats)

created topic vitals.raw (partitions=3)
created topic vitals.deadletter (partitions=1)


  DLQ p0@8: reading_id: String should have at least 1 character
  DLQ p0@10: temperature_c: Input should be less than or equal to 45
  DLQ p0@34: recorded_at: Value error, recorded_at 2026-09-11T16:08:19.382026+00:00 is in the future
  DLQ p0@44: recorded_at: Field required
  DLQ p0@57: recorded_at: Value error, recorded_at 2026-09-11T16:08:19.382026+00:00 is in the future
  DLQ p0@61: heart_rate: Input should be greater than or equal to 20
  DLQ p0@65: recorded_at: Value error, recorded_at 2026-09-11T16:08:19.382026+00:00 is in the future
  DLQ p0@66: reading_id: String should have at least 1 character
  DLQ p0@67: diagnosis: Extra inputs are not permitted
  DLQ p0@69: recorded_at: Value error, recorded_at 2026-09-11T16:08:19.383967+00:00 is in the future
  DLQ p0@70: heart_rate: Input should be less than or equal to 250
  DLQ p0@71: spo2: Input should be less than or equal to 100
  DLQ p0@74: heart_rate: Input should be greater than or equal to 20
  DLQ p0@80: reading_id: String shou

  DLQ p1@164: consciousness: Input should be 'A', 'V', 'P' or 'U'
  DLQ p1@170: reading_id: Field required
  DLQ p1@173: diagnosis: Extra inputs are not permitted
  DLQ p1@178: diagnosis: Extra inputs are not permitted
  DLQ p1@181: reading_id: Field required
  DLQ p1@183: diagnosis: Extra inputs are not permitted
  DLQ p1@185: resp_rate: Field required
{'consumed': 600, 'valid': 517, 'rejected': 83}


In [5]:
bronze = read_bronze().to_pandas()
print('Bronze rows      :', len(bronze))
print('distinct patients:', bronze.patient_id.nunique())
print('heart_rate range :', bronze.heart_rate.min(), '-', bronze.heart_rate.max())
print('spo2 range       :', bronze.spo2.min(), '-', bronze.spo2.max())
bronze.head()

Bronze rows      : 517


distinct patients: 20
heart_rate range : 56 - 104
spo2 range       : 89 - 100


,reading_id,patient_id,recorded_at,heart_rate,resp_rate,systolic_bp,spo2,temperature_c,consciousness,on_supplemental_o2,source_device,ingested_at,kafka_partition,kafka_offset
0,a558d773-90de-4dc0-a6eb-1df5b1cc02c5,P100001,2026-09-09 15:21:14.379816+00:00,88,13,110,98,36.9,A,False,Philips-IntelliVue-MX40,2026-09-09 16:08:26.578084+00:00,1,162
1,92195dfd-dc3f-42e9-9617-5c8440977fe6,P100001,2026-09-09 15:23:23.379816+00:00,84,14,113,100,36.8,A,False,Masimo-Rad-97,2026-09-09 16:08:26.578084+00:00,1,163
2,f14171f1-27ba-4761-9765-150d5db7048b,P100012,2026-09-09 15:24:49.379816+00:00,80,14,117,99,36.2,A,False,Masimo-Rad-97,2026-09-09 16:08:26.578084+00:00,1,165
3,7792da32-6ca9-4705-be5d-f54a90994949,P100019,2026-09-09 15:26:22.379816+00:00,80,13,123,99,37.0,A,False,Masimo-Rad-97,2026-09-09 16:08:26.578084+00:00,1,166
4,8005d842-cd76-471d-ab43-f3a78f282c79,P100001,2026-09-09 15:29:30.379816+00:00,90,15,112,98,36.4,A,False,Masimo-Rad-97,2026-09-09 16:08:26.578084+00:00,1,167


### Dead-letter contents - every rejected record carries its reason

In [6]:
from kafka import KafkaConsumer, TopicPartition
c = KafkaConsumer(bootstrap_servers='localhost:9092', auto_offset_reset='earliest',
                  consumer_timeout_ms=5000, value_deserializer=lambda b: json.loads(b.decode()))
tp = TopicPartition('vitals.deadletter', 0); c.assign([tp]); c.seek_to_beginning(tp)
dlq = list(c); c.close()
print('dead-letter messages:', len(dlq))
seen = set()
for m in dlq:
    key = m.value['errors'][0].split(':')[0]
    if key in seen: continue
    seen.add(key)
    print(' ', m.value['errors'][0], '| source', m.value['source'])

C:\Users\Huawei\AppData\Local\Temp\ipykernel_18504\4179376395.py:2: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  c = KafkaConsumer(bootstrap_servers='localhost:9092', auto_offset_reset='earliest',


dead-letter messages: 83
  reading_id: String should have at least 1 character | source {'topic': 'vitals.raw', 'partition': 0, 'offset': 8}
  temperature_c: Input should be less than or equal to 45 | source {'topic': 'vitals.raw', 'partition': 0, 'offset': 10}
  recorded_at: Value error, recorded_at 2026-09-11T16:08:19.382026+00:00 is in the future | source {'topic': 'vitals.raw', 'partition': 0, 'offset': 34}
  heart_rate: Input should be greater than or equal to 20 | source {'topic': 'vitals.raw', 'partition': 0, 'offset': 61}
  diagnosis: Extra inputs are not permitted | source {'topic': 'vitals.raw', 'partition': 0, 'offset': 67}
  spo2: Input should be less than or equal to 100 | source {'topic': 'vitals.raw', 'partition': 0, 'offset': 71}
  patient_id: String should match pattern '^P\d{6}$' | source {'topic': 'vitals.raw', 'partition': 2, 'offset': 1}
  consciousness: Input should be 'A', 'V', 'P' or 'U' | source {'topic': 'vitals.raw', 'partition': 2, 'offset': 40}
  resp_rate: